# 扩散语言模型与并行生成

> 写出一段 200 token 的回答，自回归模型要做 200 次前向传播，一次输出一个 token。串行是刻在 next token prediction 里的属性，GPU 算力再过剩也帮不上忙。
>
> 图像生成没有这个烦恼。Stable Diffusion 画一张图，是整张图一起从噪声里浮现，几十步迭代就完成，不需要从左上角第一个像素画到右下角。
>
> 文本能「整段一起浮现」吗？困难很具体：像素是连续的数字，可以一点点加噪、再去噪；token 是离散的类别，往一个词上「加一点噪声」没有自然的定义。这本附录就看研究者如何绕开这个障碍：用 [MASK] 充当噪声，把 diffusion 的思想搬到语言上，再动手训练一个玩具版本，实测它与自回归的差距。

## 0. 自回归的串行瓶颈

「解码策略」和「投机解码」两本都在处理同一个事实：自回归模型生成长度为 $L$ 的序列需要 $L$ 次前向传播，且必须依次进行，第 $i$ 个 token 的分布依赖前 $i-1$ 个。投机解码的思路是「一次前向验证多个候选」，平均能省一些，但它没有改变串行这个框架。

想真正摆脱串行，得换一个生成范式。

**非自回归生成（Non-Autoregressive Generation, NAR）**：不按从左到右的顺序、在一次或少数几次前向传播中同时产出全部 token 的生成方式。通俗地说，就是把「逐字往下写」换成「整段一起写，写完再检查修改」。

听起来很美，但朴素的 NAR（一次前向直接输出所有 token）在自然语言上质量很差。原因值得用手算看清。

In [ ]:
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("使用的设备：", device)

In [ ]:
# 玩具例子：某地餐厅评价的语料里只有两种固定搭配
# 「好吃不贵」和「难吃且贵」——两个空总是一起出现，彼此相关

p_slot1 = {"好": 0.5, "难": 0.5}
p_slot2_given = {
    "好": {"不": 0.9, "且": 0.1},   # 前面是「好吃」，后面大概率接「不贵」
    "难": {"且": 0.9, "不": 0.1},   # 前面是「难吃」，后面大概率接「且贵」
}

# 一步生成：两个空必须同时填，第二个空只能用边缘分布
p_slot2_marginal = {"不": 0.5, "且": 0.5}
one_shot_ok = (p_slot1["好"] * p_slot2_marginal["不"]
               + p_slot1["难"] * p_slot2_marginal["且"])

# 从左到右生成：填第二个空时已经看到第一个空
ar_ok = (p_slot1["好"] * p_slot2_given["好"]["不"]
         + p_slot1["难"] * p_slot2_given["难"]["且"])

print(f"一步生成（两个空独立填）：合法搭配概率 = {one_shot_ok:.0%}")
print(f"从左到右生成（先填第一个空）：合法搭配概率 = {ar_ok:.0%}")
print()
print("关键观察：每个空单独看都有 90% 的把握，独立填却只有 50% 合法。")
print("          一步生成的问题不在「单个词选错」，而在「组合不搭配」。")

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.2))
ax.bar(["one-shot\n(independent slots)", "left-to-right\n(AR)"],
       [one_shot_ok, ar_ok], color=["steelblue", "tomato"], width=0.5)
ax.set_ylim(0, 1)
ax.set_ylabel("probability of a valid phrase")
ax.set_title("Why one-shot generation fails: slots must coordinate")
for i, v in enumerate([one_shot_ok, ar_ok]):
    ax.text(i, v + 0.03, f"{v:.0%}", ha="center")
plt.show()

这个玩具例子里只有两个空，差距已经从 90% 掉到 50%。真实句子里互相依赖的位置有几十上百个，一步生成的「组合不搭配」会被急剧放大。

四条路线摆在一起看：

| 生成方式 | 前向传播次数 | 位置之间怎么配合 |
|:---|:---|:---|
| 自回归 | $L$ 次，串行 | 天然配合：右边总是看得到左边 |
| 投机解码 | 约 $L/\text{接受长度}$，仍是串行框架 | 配合方式同自回归 |
| 一步 NAR | 1 次 | 不配合：各填各的，容易组合失调 |
| Masked Diffusion | $k$ 次，$k$ 可调 | 部分配合：每一步都重新看到全局 |

Masked Diffusion 是 NAR 的改良版：承认一次做不完全对，那就做几轮，每轮只定下最有把握的部分，剩下的下一轮再说。它就是这本附录的主角。

## 1. 图像 diffusion 的最小心智模型

**Diffusion Model（扩散模型）**：一类定义了「逐步加噪」与「逐步去噪」两个互逆过程、通过学习去噪过程来生成的模型。通俗解释：不直接学「怎么生成」，而是学「怎么把弄乱的东西收拾回去」。会收拾，自然就会生成：从一团纯噪声开始收拾，收拾到底就是一张新图。像雕刻家说的那样，雕像本来就在石头里，要做的是把多余的部分去掉。

两个过程：

- **Forward（前向加噪）**：对真实图片不断叠加高斯噪声。噪声强度由时间步 $t$ 控制，$t=0$ 是原图，$t$ 越大越乱，$t=1$ 时几乎是纯噪声
- **Reverse（反向去噪）**：训练一个网络，输入带噪图片和 $t$，预测「干净的样子」。采样时从纯噪声出发，反复调用网络，一小步一小步走回干净图片

In [ ]:
# 用一张 24x24 的合成图像演示 forward 过程：图案逐步被噪声淹没
img = np.zeros((24, 24))
img[8:16, 4:20] = 1.0   # 横条
img[4:20, 10:14] = 1.0  # 竖条，合成一个十字

rng = np.random.default_rng(42)
fig, axes = plt.subplots(1, 5, figsize=(12, 2.6))
for ax, t in zip(axes, [0.0, 0.2, 0.5, 1.0, 2.0]):
    noisy = np.clip(img + rng.normal(0, t, img.shape), 0, 1)
    ax.imshow(noisy, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"noise level t = {t:.1f}")
    ax.axis("off")
plt.suptitle("Forward process: an image is gradually destroyed by noise", y=1.04)
plt.show()

print("关键观察：reverse 过程就是从最右边走回最左边。")
print("          每一步的更新是全图同时进行的，去噪网络看得到当前整张图。")

并行性在这里不是优化技巧，而是这种生成方式的固有属性：每一步、每个像素的更新都基于全局信息。

这个性质对文本至关重要。回忆 Self-Attention 一节：去掉 causal mask，每个位置就能看到所有位置。双向注意力正是文本版 diffusion 的地基。

## 2. 把噪声换成 [MASK]

图像能加高斯噪声，因为像素是连续的数字，加完还是数字。token 不行：词表是一个离散集合，给 `"猫"` 加上 0.1 倍的噪声，结果不在词表里，这个操作没有定义。

**Masked Diffusion（掩码扩散）**：把「加噪」定义为「以一定概率把 token 替换成特殊符号 [MASK]」、把「去噪」定义为「预测被 [MASK] 位置的原 token」的扩散模型。通俗解释：把一句话随机挖空，模型来填空；挖得越狠对应噪声越强；全部挖空就是纯噪声。

对照图像 diffusion 的两个过程：

- **Forward**：每个位置独立地以概率 $t$ 被替换成 [MASK]。$t=0$ 是原句，$t=1$ 是一整行 [MASK]
- **Reverse**：训练一个双向模型，输入带 [MASK] 的句子，在每个 [MASK] 位置输出词表上的概率分布

In [ ]:
# 演示 forward 过程作用在文本上的样子：█ 代表 [MASK]
demo_text = "12+34=046"
rng = np.random.default_rng(42)

fig, axes = plt.subplots(1, 4, figsize=(11, 2.0))
for ax, t in zip(axes, [0.25, 0.5, 0.75, 1.0]):
    shown = [c if rng.random() > t else "█" for c in demo_text]
    for col, ch in enumerate(shown):
        if ch == "█":
            face, edge = "#cccccc", "none"
        else:
            face, edge = "#e8f0fe", "#7a9cc6"
        ax.text(col, 0, ch, ha="center", va="center", fontsize=13,
                bbox=dict(boxstyle="square,pad=0.4", facecolor=face, edgecolor=edge))
    ax.set_xlim(-0.8, len(demo_text) - 0.2)
    ax.set_ylim(-0.6, 0.6)
    ax.axis("off")
    ax.set_title(f"mask rate t = {t:.2f}", fontsize=10)
plt.suptitle("Forward process on text: masking plays the role of noise", y=1.12)
plt.show()

print("关键观察：t 从小到大，句子逐渐被 [MASK] 淹没；t=1 时信息全部丢失。")
print("          生成就是反着走：从全 [MASK] 出发，一步步把信息恢复出来。")

你大概已经察觉到了什么——这个训练目标和 BERT 一节的 MLM（Masked Language Modeling）几乎是同一件事。确实如此，区别在于用法：

| | BERT 的 MLM | Masked Diffusion |
|:---|:---|:---|
| 训练时 mask 比例 | 固定 15% | 从 0 到 100% 均匀采样 |
| 预测几次 | 一次 | 采样时迭代多轮 |
| 注意力 | 双向 | 双向 |

BERT 挖一次空、填一次，是个理解型任务；diffusion 把「挖空与填补」变成一条可以走的路径，从全空走到全满，这才是生成。

**反向过程（生成）**从全 [MASK] 的答案区开始，每轮三步：

1. 把当前序列喂给模型，拿到每个 [MASK] 位置的概率分布
2. 每个位置的最大概率是它的**置信度**：模型对这个位置有多确定
3. 揭晓置信度最高的 $k$ 个位置，其余保持 [MASK]，进入下一轮

这就是**置信度优先解码**：先做有把握的决定，把最不确定的位置留到信息更多的后面几轮。工业界把「每步选哪些位置揭晓」统称为 remasking 策略，置信度优先只是其中一种，第 6 节会看到全家福。

用考试类比：先把会的题写了，不确定的先空着；写完一遍，新写下的答案会提供新线索，回头再攻空题。

## 3. 手算验证：一次 3 步生成

规则先行：**每步揭晓几个**。设还剩 $m$ 个 [MASK]、还剩 $s$ 步（含当前步），每步揭晓 $k=\lceil m/s \rceil$ 个。向上取整保证恰好走完。

任务：把 `9 2 5 1` 排序。完整序列是 `9 2 5 1=1 2 5 9`，`=` 之后是 7 个字符的答案区（4 个数字 + 3 个空格）。用 $T=3$ 步生成，输入区（`=` 之前）始终可见。

**第 1 步**：7 个位置全是 [MASK]，$k=\lceil 7/3\rceil=3$。模型是双向的，看得到输入区的 `9 2 5 1`：

| 位置 | 内容 | 模型的概率（示意） | 置信度 |
|:--|:--|:--|:--|
| 0 | [MASK] | `1`: 0.60，`2`: 0.25，其他 0.15 | 0.60 |
| 1 | [MASK] | `空格`: 0.98，`5`: 0.01，… | **0.98** |
| 2 | [MASK] | `2`: 0.45，`5`: 0.40，… | 0.45 |
| 3 | [MASK] | `空格`: 0.98，… | **0.98** |
| 4 | [MASK] | `5`: 0.45，`2`: 0.40，… | 0.45 |
| 5 | [MASK] | `空格`: 0.98，… | **0.98** |
| 6 | [MASK] | `9`: 0.75，`5`: 0.15，… | 0.75 |

空格位置的排列模式固定，置信度接近 1，最优先揭晓。数字里最小值 `1` 和最大值 `9` 相对好认，中间的 `2` `5` 最模糊。揭晓位置 1、3、5，答案区变成 `[M]空[M]空[M]空[M]`。

**第 2 步**：剩 4 个数字位，$k=\lceil 4/2\rceil=2$。注意概率变了：空格已定，模式更清晰：

| 位置 | 概率（示意） | 置信度 | |
|:--|:--|:--|:--|
| 0 | `1`: 0.90，… | **0.90** | 揭晓 |
| 2 | `2`: 0.50，`5`: 0.42，… | 0.50 | 保留 |
| 4 | `5`: 0.50，`2`: 0.42，… | 0.50 | 保留 |
| 6 | `9`: 0.88，… | **0.88** | 揭晓 |

**第 3 步**：剩位置 2、4，$k=\lceil 2/1\rceil=2$。此刻两端 `1` 和 `9` 已揭晓，输入里剩下的数字只有 `2` 和 `5`，顺序被锁死：

| 位置 | 概率（示意） | 置信度 | |
|:--|:--|:--|:--|
| 2 | `2`: 0.95，… | **0.95** | 揭晓 |
| 4 | `5`: 0.97，… | **0.97** | 揭晓 |

手算里最重要的一个观察：**位置 2 的置信度从第 1 步的 0.45 涨到了第 3 步的 0.95**。它自己没有变，变的是它的上下文：每轮揭晓的 token 都成了新的条件。迭代去噪的价值就在这里，把最难的决定推迟到信息最多的时刻。

反过来也看清了 1 步生成的处境：模型必须在信息最少的时候（全 [MASK]）做全部决定。这正是下一节实验里 1 步质量崩塌的原因。

## 4. 代码实现：Mini Masked Diffusion LM

任务选**排序**：输入 8 个 0-9 的随机数字，输出升序排列。完整序列形如 `8 6 5 2 3 0 0 0=0 0 0 2 3 5 6 8`。选它的理由有三条：

- 答案区长达 15 个 token，自回归要 15 次前向，并行生成的收益看得见
- 对不对可以自动判定：整条序列和标准答案完全相同才算对（exact match）
- 排序需要全局信息，每个输出位置都依赖整个输入，双向注意力真的在干活

模型直接复用「从零实现 GPT」一节的结构，只有一处差别：**去掉 causal mask**。另外我们不给模型注入时间步 $t$ 的 embedding，玩具任务上模型自己能应付不同的 mask 比例；真实模型的做法见第 6 节。

In [ ]:
def make_sample(rng, n_nums=8):
    """随机生成 n_nums 个 0-9 的数字，返回「乱序=升序」格式的字符串"""
    nums = list(rng.integers(0, 10, size=n_nums))
    left = " ".join(str(d) for d in nums)
    right = " ".join(str(d) for d in sorted(nums))
    return f"{left}={right}"

CHARS = sorted(set("0123456789 ="))
char2id = {c: i for i, c in enumerate(CHARS)}
MASK_ID = len(CHARS)           # [MASK] 排在词表最后
VOCAB = len(CHARS) + 1

def encode(s):
    return [char2id[c] for c in s]

def decode(ids):
    """把 id 序列还原成字符串；[MASK] 显示成 █"""
    table = {i: c for c, i in char2id.items()}
    return "".join(table.get(i, "█") for i in ids)

rng = np.random.default_rng(1)
data = torch.tensor([encode(make_sample(rng)) for _ in range(20000)])

SEQ_LEN = data.shape[1]         # 8 个数字 + 7 个空格 + = + 15 个答案字符 = 31
ANS_START = 16                  # '=' 之后第一个位置，从这里开始是答案区
ANS_LEN = SEQ_LEN - ANS_START   # 15

print("样例：", decode(data[0].tolist()))
print(f"序列长度 {SEQ_LEN}，答案区 {ANS_LEN} 个位置，词表大小 {VOCAB}（含 [MASK]）")
print("关键观察：答案区 15 个位置，自回归要 15 次前向；下面看 diffusion 需要几次")

模型定义。与「从零实现 GPT」一节的 Mini-GPT 相比，唯一改动是 `bidirectional` 参数：为 `True` 时不传 causal mask，每个位置互相可见。

In [ ]:
class Block(nn.Module):
    """标准 Transformer Block：Pre-LN + Self-Attention + MLP"""

    def __init__(self, d, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_head, batch_first=True)
        self.ln2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(
            nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d)
        )

    def forward(self, x, attn_mask):
        h = self.ln1(x)
        a, _ = self.attn(h, h, h, attn_mask=attn_mask)
        x = x + a
        return x + self.mlp(self.ln2(x))


class TinyTransformer(nn.Module):
    """小 Transformer。bidirectional=True 时无 causal mask（diffusion 用），
    False 时加 causal mask（自回归基线用）"""

    def __init__(self, d=128, n_layer=3, n_head=4, bidirectional=True):
        super().__init__()
        self.tok = nn.Embedding(VOCAB, d)
        self.pos = nn.Embedding(40, d)
        self.blocks = nn.ModuleList([Block(d, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, VOCAB)
        self.bidirectional = bidirectional

    def forward(self, idx):
        T = idx.shape[1]
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        if self.bidirectional:
            mask = None
        else:
            # 上三角为 True = 禁止看向未来：这就是 causal mask
            mask = torch.triu(
                torch.ones(T, T, dtype=torch.bool, device=idx.device), diagonal=1
            )
        for b in self.blocks:
            x = b(x, mask)
        return self.head(self.ln_f(x))

In [ ]:
torch.manual_seed(42)
diff_model = TinyTransformer(bidirectional=True).to(device)
print("参数量：", sum(p.numel() for p in diff_model.parameters()))

训练循环就是「挖空与补空」，与 BERT 的差别只在 mask 比例是随机的：

1. 每条序列采样一个噪声水平 $t \sim U(0,1)$
2. 每个位置独立地以概率 $t$ 被替换成 [MASK]
3. 交叉熵只在被 mask 的位置计算，模型学的就是「看上下文补空」

在 Apple Silicon GPU 上训练约 6 分钟，纯 CPU 会明显更久。等待的时间正好可以回味第 3 节的手算。

In [ ]:
def train_diffusion(model, steps=3000, bs=128, lr=1e-3):
    """训练 masked diffusion 模型。返回 (step, loss) 列表用于画曲线"""
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    hist = []
    for step in range(steps):
        batch = data[torch.randint(0, len(data), (bs,))].to(device)
        t = torch.rand(bs, 1, device=device)               # 每条序列一个噪声水平
        mask = torch.rand(bs, SEQ_LEN, device=device) < t  # True 的位置将被挖空
        if not mask.any():
            continue                                        # 小概率一个都没挖到
        x = batch.clone()
        x[mask] = MASK_ID
        logits = model(x)
        loss = F.cross_entropy(logits[mask], batch[mask])   # 只在挖空处算损失
        opt.zero_grad()
        loss.backward()
        opt.step()
        if (step + 1) % 500 == 0:
            print(f"  step {step + 1:5d}  loss {loss.item():.4f}")
            hist.append((step + 1, loss.item()))
    return hist

In [ ]:
t0 = time.time()
diff_hist = train_diffusion(diff_model, steps=3000)
print(f"训练完成，用时 {time.time() - t0:.0f} 秒")

接下来实现解码循环。对照第 3 节的手算：预测所有 [MASK] 位置、按置信度揭晓 top-$k$、其余留到下一轮。

In [ ]:
@torch.no_grad()
def diffuse_generate(model, prompt_ids, n_steps, record=False):
    """从「答案区全 [MASK]」开始做 n_steps 步去噪。
    record=True 时额外返回每步的序列快照，用于可视化"""
    model.eval()
    seq = list(prompt_ids) + [MASK_ID] * ANS_LEN
    masked = list(range(ANS_START, SEQ_LEN))
    snaps = [list(seq)]
    for step_i in range(n_steps):
        ids = torch.tensor([seq], device=device)
        logits = model(ids)[0]
        pos = torch.tensor(masked, device=device)
        probs = F.softmax(logits[pos], dim=-1)
        conf, pick = probs.max(dim=-1)                   # 最大概率 = 置信度
        k = math.ceil(len(masked) / (n_steps - step_i))  # 保证 n_steps 步内走完
        top = conf.argsort(descending=True)[:k]          # 置信度最高的 k 个
        for j in top.tolist():
            seq[masked[j]] = pick[j].item()
        chosen = set(top.tolist())
        masked = [masked[j] for j in range(len(masked)) if j not in chosen]
        snaps.append(list(seq))
    assert not masked, "解码结束后不应剩任何 [MASK]"
    return (seq, snaps) if record else seq

In [ ]:
# 固定一批测试样本，后面所有评测都用它们
test_rng = np.random.default_rng(99)
tests = [make_sample(test_rng) for _ in range(200)]

sample = tests[0]
prompt = encode(sample[:ANS_START])
print("输入（可见部分）：", sample[:ANS_START])
print("标准答案：", sample[ANS_START:])
seq, snaps = diffuse_generate(diff_model, prompt, n_steps=5, record=True)
print("生成结果：", decode(seq)[ANS_START:])
print("整条全对：", decode(seq) == sample)
print()
for i, snap in enumerate(snaps):
    label = "初始" if i == 0 else f"第 {i} 步"
    print(f"  {label}: {decode(snap)[ANS_START:]}")

## 5. 实验观察

先把刚才 5 步生成的过程画出来：每一行是一个解码时刻，每一列是答案区的一个位置，█ 是还没揭晓的 [MASK]。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
grid = np.zeros((len(snaps), ANS_LEN))
for r, snap in enumerate(snaps):
    for c, ch in enumerate(decode(snap)[ANS_START:]):
        grid[r, c] = 0.0 if ch == "█" else 1.0
ax.imshow(grid, cmap="Blues", aspect="auto", vmin=0, vmax=1)
for r, snap in enumerate(snaps):
    for c, ch in enumerate(decode(snap)[ANS_START:]):
        ax.text(c, r, ch, ha="center", va="center", fontsize=10,
                color="gray" if ch == "█" else "black")
ax.set_yticks(range(len(snaps)))
ax.set_yticklabels(["init"] + [f"step {i + 1}" for i in range(len(snaps) - 1)])
ax.set_xlabel("answer position")
ax.set_title("Masked diffusion decoding: positions get revealed over steps")
plt.show()
print("关键观察：空格位置最先被填上（最容易），数字按置信度逐步揭晓。")
print("          每一步都在「整段一起写」，而不是从左到右逐字符写。")

再训一个自回归基线做对照。同一个 TinyTransformer，`bidirectional=False`，训练目标是标准的 next token prediction；生成时从左到右，每个 token 一次前向。

In [ ]:
def train_ar(model, steps=3000, bs=128, lr=1e-3):
    """训练自回归基线：全部位置都算 next token 的交叉熵"""
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    hist = []
    for step in range(steps):
        batch = data[torch.randint(0, len(data), (bs,))].to(device)
        logits = model(batch[:, :-1])
        loss = F.cross_entropy(
            logits.reshape(-1, VOCAB), batch[:, 1:].reshape(-1)
        )
        opt.zero_grad()
        loss.backward()
        opt.step()
        if (step + 1) % 500 == 0:
            print(f"  step {step + 1:5d}  loss {loss.item():.4f}")
            hist.append((step + 1, loss.item()))
    return hist


@torch.no_grad()
def ar_generate(model, prompt_ids):
    """自回归生成：每次前向只为拿下一个 token，共 ANS_LEN 次前向"""
    model.eval()
    seq = list(prompt_ids)
    for _ in range(ANS_LEN):
        ids = torch.tensor([seq], device=device)
        seq.append(int(model(ids)[0, -1].argmax()))
    return seq

In [ ]:
torch.manual_seed(42)
ar_model = TinyTransformer(bidirectional=False).to(device)
t0 = time.time()
ar_hist = train_ar(ar_model, steps=3000)
print(f"训练完成，用时 {time.time() - t0:.0f} 秒")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
dx, dy = zip(*diff_hist)
ax_, ay = zip(*ar_hist)
ax.plot(dx, dy, "o-", ms=4, label="masked diffusion (loss on masked positions)")
ax.plot(ax_, ay, "s-", ms=4, label="autoregressive (loss on all positions)")
ax.set_xlabel("training step")
ax.set_ylabel("cross-entropy loss")
ax.set_title("Both objectives converge")
ax.legend()
plt.show()
print("关键观察：两条 loss 各自趋平即收敛。数值不能横向比：")
print("          diffusion 的 loss 只算被挖空的位置，天然包含高噪声的难题")

正戏：**步数与质量的关系**。对同一批 200 个测试样本，分别用 1、2、3、5、8、15 步生成，统计整条序列全对的比例；自回归基线固定 15 次前向。

In [ ]:
def acc_diffusion(n_steps, n=200):
    ok = 0
    for s in tests[:n]:
        gen = diffuse_generate(diff_model, encode(s[:ANS_START]), n_steps)
        ok += (decode(gen) == s)
    return ok / n


def acc_ar(n=200):
    ok = 0
    for s in tests[:n]:
        gen = ar_generate(ar_model, encode(s[:ANS_START]))
        ok += (decode(gen) == s)
    return ok / n


step_list = [1, 2, 3, 5, 8, 15]
diff_accs = [acc_diffusion(ns) for ns in step_list]
ar_acc = acc_ar()

for ns, a in zip(step_list, diff_accs):
    print(f"diffusion {ns:2d} 步：整条全对率 {a:.1%}")
print(f"自回归    {ANS_LEN} 步：整条全对率 {ar_acc:.1%}")
print()
print(f"关键观察：{step_list[0]} 步只有 {diff_accs[0]:.0%}，"
      f"{step_list[-1]} 步升到 {diff_accs[-1]:.0%}——步数就是质量旋钮")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))

ax1.plot(step_list, diff_accs, "o-", label="masked diffusion")
ax1.axhline(ar_acc, color="tomato", ls="--",
            label=f"autoregressive ({ANS_LEN} fwd passes)")
ax1.set_xlabel("number of diffusion steps")
ax1.set_ylabel("exact match accuracy")
ax1.set_title("Quality vs number of steps")
ax1.set_ylim(0, 1.05)
ax1.legend()

labels = ["AR"] + [f"{ns}-step" for ns in step_list]
fwd = [ANS_LEN] + step_list
ax2.bar(labels, fwd, color=["tomato"] + ["steelblue"] * len(step_list))
ax2.set_ylabel("forward passes per generation")
ax2.set_title("Cost: forward passes")
plt.tight_layout()
plt.show()

三个读数（下面引用的百分比来自本次实际运行，你复跑时数字会浮动，趋势应当一致）：

- **1 步崩塌**：约 38% 对自回归的 100%。注意这个任务是确定性的，给定输入只有唯一正确答案，所以这不是第 0 节那种「组合不搭配」，而是模型容量有限时一次前向做不完全部推理，又没有机会修正
- **步数是质量旋钮**：1 到 15 步，正确率单调上升。要 5 倍少的前向？用 3 步，接受一半的正确率。要质量？加步数。这种「生成时才决定的档位」是自回归没有的自由度
- **全步数仍逊于自回归**：15 步 diffusion 与自回归的前向次数相同，质量仍差一截。这与工业界的公开结果一致：diffusion LM 追近但尚未追平同规模 AR 模型

| | 自回归 | 一步 NAR | Masked Diffusion |
|:---|:---|:---|:---|
| 前向次数 | $L$，串行 | 1 | $k$，可调 |
| 位置配合 | 从左到右条件生成 | 无 | 每步全局重看 |
| 质量上限 | 最高 | 最低 | 接近但低于自回归 |
| 独有自由度 | 无 | 无 | 用步数换速度的连续旋钮 |

## 6. 和工业界的区别

**真实模型长什么样。** 代表作是 LLaDA（2025，8B）：从 LLaMA 3 初始化继续训练成 masked diffusion 模型，在不少通用基准上接近同规模的 AR 模型。它的两处升级，玩具版都省了：

- 训练目标不是朴素交叉熵，而是带似然权重的 ELBO，不同噪声水平 $t$ 的贡献按理论加权
- 把 $t$ 的 embedding 注入模型，让模型明确知道「现在噪声多大」（我们靠模型自己适应）

**remasking 策略全家福：**

| 策略 | 每步揭晓谁 | 一句话点评 |
|:---|:---|:---|
| random | 随机挑 | 最简单，质量最差 |
| confidence | 置信度最高的 | 本附录的实现，最常用 |
| semi-autoregressive | 从左到右按块揭晓 | 保留 AR 的位置感 |
| low-confidence remasking | 每步把低置信度的已揭晓位置重新变回 [MASK] | 允许反悔，质量更高 |

我们实现的版本「揭晓不反悔」；最后一种每轮都会把模型没把握的位置重新挖空，下一轮重新决定。改错的能力是 diffusion 相对自回归的一个结构性优势：AR 生成错了只能错下去，这与「后训练技术演进」一节讨论的 exposure bias 同根同源。

**推理系统要重新设计。** KV cache 对 diffusion 失效：双向注意力加每步整段重算，缓存的结构不存在了。「现代 LLM 推理系统」一节里的 continuous batching、调度思想可以复用，但瓶颈和优化点都不一样。

**现状与定位。** 2025 年起有了商业落地：Inception Labs 的 Mercury 主打代码生成（官方数字超过 1000 tokens/s），Google 发布了 Gemini Diffusion。公开评测里 diffusion LM 的质量仍普遍逊于同规模 AR，在代码这类输出分布集中（低熵）的任务上优势最明显。缩放定律、对齐方法、推理引擎这套生态都围绕 AR 建立，迁移需要时间；不少研究者认为 AR 打底、diffusion 加速的混合架构是过渡期的形态。

它值不值得学？回到开头：串行瓶颈是 next token prediction 的结构性约束，diffusion 是目前唯一走进工业界的「换范式」答案。理解它，你就多了一个自回归之外的参照系。

## 小结

- 自回归的串行瓶颈：$L$ 个 token 要 $L$ 次依序前向；投机解码是框架内的缓解，diffusion 是换框架
- 一步 NAR 失败的根源是「组合不搭配」：每个位置各自都有把握，合起来却不合法
- Masked Diffusion 用 [MASK] 充当噪声：加噪是按比例挖空，去噪是双向模型补空；训练目标与 BERT 的 MLM 同源，mask 比例覆盖 0 到 100%
- 解码循环三步：预测所有 [MASK] 位置、按置信度揭晓 top-$k$、其余留到下一轮；$k=\lceil m/s\rceil$ 保证收尾
- 实验结论：步数是质量旋钮，单调地换质量；全步数仍略逊自回归，与工业界观察一致
- 工业坐标：LLaDA、Mercury、Gemini Diffusion；remasking 策略家族；KV cache 失效带来的推理系统重构

想继续深入：LLaDA 论文附录里有完整的 ELBO 推导；把本附录的 `diffuse_generate` 改成 low-confidence remasking 版本，是检验你是否真懂解码循环的最好练习。

## 作业

> 可以用 AI 询问思路、拆步骤、检查方向，但不建议直接让 AI 「做完这道题」。

三道题都基于已训练好的模型和第 4 节的代码，不需要重新训练。

1. **实现前向过程的 mask 采样**。小提示：每个位置独立判断，一次 `torch.rand` 加一个比较运算符即可
2. **每步揭晓几个位置**。小提示：15 个位置分 4 步，想想用 floor 会在哪一步出问题
3. **random 揭晓对比 confidence 揭晓**。小提示：`torch.randperm` 给出均匀随机排列

In [ ]:
# 作业 1：实现前向过程的 mask 采样
# 每个位置独立地以概率 t 被 mask——这是 masked diffusion 对「加噪」的定义

def sample_mask(seq_len, t, gen):
    """返回布尔张量：True 表示该位置要替换成 [MASK]
    seq_len：序列长度；t：噪声水平（0~1）；gen：随机数生成器（保证可复现）
    """
    rand = torch.rand(seq_len, generator=gen)
    mask = rand ___ t          # ← 填空：一个比较运算符
    return mask.bool()

g = torch.Generator().manual_seed(0)
m = sample_mask(1000, 0.3, g)
assert m.dtype == torch.bool, "mask 应该是布尔张量"
assert 200 <= int(m.sum()) <= 400, \
    f"t=0.3、长度 1000 期望约 300 个 True，你得到 {int(m.sum())}"
assert sample_mask(1000, 1.0, torch.Generator().manual_seed(1)).all(), \
    "t=1.0 时应该全部被 mask"

print("✅ 作业 1 通过：你已经会给序列采样噪声了")
print("   t 就是 mask 比例：t 越大噪声越强，t=1 时整条序列变成 [MASK]")

In [ ]:
# 作业 2：每步揭晓多少个位置？
# k 取小了走不完，取大了浪费步数

def reveal_count(remaining, steps_left):
    """remaining：还剩多少个 [MASK]；steps_left：还剩多少步（含当前步）
    返回这一步要揭晓的位置数
    """
    return math.___(remaining / steps_left)   # ← 填空：ceil 还是 floor？

# 15 个位置、4 步走完：每步 4 4 4 3
assert reveal_count(15, 4) == 4
assert reveal_count(11, 3) == 4
assert reveal_count(3, 1) == 3

print("✅ 作业 2 通过：向上取整保证恰好走完")
print("   如果用 floor：15//4=3，四步只揭晓 12 个，最后 3 个位置没有步数可用了")

In [ ]:
# 作业 3：把 confidence 揭晓换成 random 揭晓，质量会掉多少？

@torch.no_grad()
def diffuse_generate_random(model, prompt_ids, n_steps, seed=7):
    """与 diffuse_generate 逻辑相同，但每步「随机」挑 k 个位置揭晓"""
    gen = torch.Generator().manual_seed(seed)
    model.eval()
    seq = list(prompt_ids) + [MASK_ID] * ANS_LEN
    masked = list(range(ANS_START, SEQ_LEN))
    for step_i in range(n_steps):
        ids = torch.tensor([seq], device=device)
        logits = model(ids)[0]
        pos = torch.tensor(masked, device=device)
        probs = F.softmax(logits[pos], dim=-1)
        _, pick = probs.max(dim=-1)
        k = math.ceil(len(masked) / (n_steps - step_i))
        top = torch.___(len(masked), generator=gen)[:k].tolist()  # ← 填空
        for j in top:
            seq[masked[j]] = pick[j].item()
        chosen = set(top)
        masked = [masked[j] for j in range(len(masked)) if j not in chosen]
    return seq

n_eval, n_steps = 100, 3
ok_r = ok_c = 0
for s in tests[:n_eval]:
    prompt = encode(s[:ANS_START])
    ok_r += (decode(diffuse_generate_random(diff_model, prompt, n_steps)) == s)
    ok_c += (decode(diffuse_generate(diff_model, prompt, n_steps)) == s)
acc_random, acc_conf = ok_r / n_eval, ok_c / n_eval
print(f"random 揭晓      3 步：正确率 {acc_random:.0%}")
print(f"confidence 揭晓  3 步：正确率 {acc_conf:.0%}")
assert acc_random <= acc_conf + 0.15, \
    "随机揭晓通常不会好于置信度优先（留了少量随机波动的余地）"

print("✅ 作业 3 通过：揭晓顺序本身是有信息量的")
print("   confidence 把容易的决定先做掉，把难的位置留到上下文最全的最后一轮")
print("   remasking 策略研究的核心问题：同样数量的前向传播，怎么分配最划算")

## 参考资料

- Ho et al., 2020, *Denoising Diffusion Probabilistic Models*（DDPM，图像 diffusion 的奠基工作）
- Gu et al., 2018, *Non-Autoregressive Neural Machine Translation*（NAR 生成的起点，组合不搭配问题首次被系统研究）
- Nie et al., 2025, *Large Language Diffusion Models*（LLaDA，本附录主要取材的对象）
- Inception Labs *Mercury* 与 Google *Gemini Diffusion* 的官方发布（2025，工业界 diffusion LM）
- Stanford CME295 (Autumn 2025) Lecture 9 与 Stanford CS336 (Spring 2025) Lecture 10 的相关讲义